
# Markov Chains in Python — Bayesian Dojo (Colab Edition)

This notebook is a Python/Colab port of a classic MATLAB teaching demo
(`markov_chain_studentdavetutorials.m` + `draw_states*.m` + `arrow3.m`) that
uses a "ninja fighting styles" story to teach **Markov chains**.

Everything here runs natively in Google Colab — no toolboxes, no `.m` files,
just `numpy`, `matplotlib`, and `networkx` (all pre-installed on Colab).

**What changed vs. the original MATLAB code:**
- The four hand-written `draw_states.m` / `draw_states3.m` / `draw_states4.m`
  functions (each hard-coded for a specific number of states, with manually
  placed `text()` labels and `arrow3()` arrows) are replaced by **one generic
  function**, `plot_markov_diagram`, that works for *any* number of states
  using `networkx`.
- The MATLAB `pause` loop (which waits for a keypress between frames) becomes
  an `IPython.display.clear_output` animation loop, which is the standard way
  to animate frame-by-frame in a Jupyter/Colab notebook.
- `P^i` (MATLAB matrix power) becomes `np.linalg.matrix_power(P, i)`.
- The eigen-decomposition trick for the stationary distribution becomes
  `np.linalg.eig`.

Run the cells top to bottom. Each "experiment" section is independent.


## Setup — generic plotting helpers

This replaces `arrow3.m`, `draw_states.m`, `draw_states3.m`, and `draw_states4.m` with one reusable set of functions.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import clear_output
import time

def plot_markov_diagram(ax, M, node_values, labels=None, cmap="Blues", title=""):
    """Draw a directed-graph view of a matrix M (e.g. P or P^i).

    node_values : values used to color/label each node (e.g. diag(P^i),
                  or a probability distribution vector)
    Replaces draw_states.m / draw_states3.m / draw_states4.m generically
    for any number of states.
    """
    n = M.shape[0]
    if labels is None:
        labels = [f"S{i}" for i in range(n)]

    G = nx.DiGraph()
    for i in range(n):
        G.add_node(i)
    for i in range(n):
        for j in range(n):
            if M[i, j] > 1e-9:
                G.add_edge(i, j, weight=M[i, j])

    pos = nx.circular_layout(G)
    cmap_obj = plt.get_cmap(cmap)
    vals = np.asarray(node_values, dtype=float)
    vmax = max(vals.max(), 1e-9)
    node_colors = cmap_obj(0.25 + 0.65 * (vals / vmax))

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1900, node_color=node_colors,
                            edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        labels={i: f"{labels[i]}\n{node_values[i]:.2f}" for i in range(n)},
        font_size=9)

    self_loops = [(u, v) for u, v in G.edges() if u == v]
    normal_edges = [(u, v) for u, v in G.edges() if u != v]

    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=normal_edges,
                            connectionstyle="arc3,rad=0.15", arrowsize=15,
                            width=1.4, edge_color="gray")
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=self_loops,
                            connectionstyle="arc3,rad=0.6", arrowsize=15,
                            width=1.4, edge_color="gray")

    edge_labels = {(u, v): f"{M[u, v]:.2f}" for u, v in G.edges()}
    nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=edge_labels, font_size=8)
    ax.set_title(title)
    ax.axis("off")


def plot_evolution(ax, i_all, t_all):
    """t_all: shape (num_matrix_entries, num_steps). Mirrors the MATLAB
    'evolution of transition probs. for each element' subplot."""
    for row in range(t_all.shape[0]):
        ax.plot(i_all, t_all[row], marker=".", linewidth=1)
    ax.set_xlabel("discrete time steps")
    ax.set_ylabel("probability")
    ax.set_title("evolution of transition probs. for each element")


def run_markov_demo(P, n_steps=40, labels=None, pause=0.12, animate=True):
    """Loops i = 1..n_steps, computes P^i, and shows:
      left  = graph view of P^i (node = diagonal 'return' probabilities)
      right = evolution of every entry of P^i over time
    This is the direct Python equivalent of the MATLAB for-loop:
        for i = 1:100
            t = P^i;
            draw_states(t,i); plot(...); pause
        end
    """
    n = P.shape[0]
    t_all = []
    i_all = []
    for i in range(1, n_steps + 1):
        Pi = np.linalg.matrix_power(P, i)
        t_all.append(Pi.flatten())
        i_all.append(i)

        if animate:
            clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        plot_markov_diagram(axes[0], Pi, np.diag(Pi), labels=labels, title=f"P^{i}")
        plot_evolution(axes[1], i_all, np.array(t_all).T)
        plt.tight_layout()
        plt.show()
        if animate:
            time.sleep(pause)

    return np.array(t_all).T, np.array(i_all)


def stationary_distribution(P):
    """Eigenvector method: replaces the MATLAB eig(P') + find-eigenvalue-1 code."""
    evals, evecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(evals - 1))
    vec = np.real(evecs[:, idx])
    return vec / vec.sum()



## Experiment 1 — a non-regular (periodic) chain

$$P = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$$

This chain just flips between the two states forever — it **never settles down**
(it's ergodic in the loose sense of visiting every state, but it is *periodic*,
so $P^i$ oscillates instead of converging). Watch the right-hand plot: the
probabilities never stop oscillating between 0 and 1.

*(Set `animate=False` if you just want the final static frame instead of a live animation.)*


In [ ]:
P = np.array([[0., 1.],
              [1., 0.]])

t_all, i_all = run_markov_demo(P, n_steps=20, labels=["A", "B"], animate=True)



## Experiment 2 — a regular chain with a zero entry

$$P = \begin{bmatrix} 1/2 & 1/2 \\ 1 & 0 \end{bmatrix}$$

Even though $P$ itself has a zero entry, some power of $P$ has **all positive
entries** — that's the definition of a *regular* chain. Regular chains always
converge to a unique stationary distribution, regardless of where you start.
Watch the diagram settle down as $i$ grows.


In [ ]:
P = np.array([[0.5, 0.5],
              [1.0, 0.0]])

t_all, i_all = run_markov_demo(P, n_steps=20, labels=["A", "B"], animate=True)

print("Stationary distribution:", stationary_distribution(P))



## Experiment 3 — "Know your opponent": Frequentisian Ninja fighting styles

Three-state chain: **Punch**, **Kick**, **Falcon Punch**. Each "style" below is
a transition matrix describing how likely a fighter is to repeat an attack
(`a`, `b`, `c` on the diagonal) versus switch to one of the other two moves.

Try each style and watch how fast (and to what distribution) it converges.
The more lopsided the diagonal, the more "predictable" (streaky) the fighter is.


In [ ]:
def ninja_style(a, b, c):
    return np.array([
        [a,       (1 - a) / 2, (1 - a) / 2],
        [(1 - b) / 2, b,       (1 - b) / 2],
        [(1 - c) / 2, (1 - c) / 2, c],
    ])

styles = {
    "E. Honda style (likes to punch)":  ninja_style(0.9, 0.3, 0.2),
    "Chun-Li style (likes to kick)":    ninja_style(0.1, 0.9, 0.3),
    "Captain Falcon style (obvious)":   ninja_style(0.1, 0.2, 0.9),
    "Master Frequentisian (balanced)":  ninja_style(1/3, 1/3, 1/3),
}

labels3 = ["Punch", "Kick", "Falcon Punch"]


In [ ]:
# Pick one style to animate (change the key to try the others)
P = styles["Master Frequentisian (balanced)"]

t_all, i_all = run_markov_demo(P, n_steps=25, labels=labels3, animate=True)

print("Stationary distribution:", stationary_distribution(P))


In [ ]:
# Compare all four styles' stationary distributions at a glance (static, no animation)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (name, P) in zip(axes, styles.items()):
    stat = stationary_distribution(P)
    plot_markov_diagram(ax, P, stat, labels=labels3, title=name)
plt.tight_layout()
plt.show()



## Experiment 4 — the 3-hit combo (absorbing-flavored 4-state chain)

The Master Ninja lands a scripted combo: **Punch → Kick → Falcon Punch → KO**.
`b` is the Bayesian Ninja's chance to *interrupt* the combo at each step and
reset it back to the start.

$$P = \begin{bmatrix}
1-a & a & 0 & 0\\
b & 0 & 1-b & 0\\
b & 0 & 0 & 1-b\\
1 & 0 & 0 & 0
\end{bmatrix}$$


In [ ]:
a = 0.5
b = 0.7  # interrupt probability

P = np.array([
    [1 - a, a,     0,     0],
    [b,     0,     1 - b, 0],
    [b,     0,     0,     1 - b],
    [1,     0,     0,     0],
])

labels4 = ["Start", "Punch", "Kick", "Falcon Punch (KO)"]

t_all, i_all = run_markov_demo(P, n_steps=25, labels=labels4, animate=True)


### Solve for the stationary distribution with eigen-decomposition (matches the MATLAB `eig(P')` block)

In [ ]:
evals, evecs = np.linalg.eig(P.T)
print("Eigenvalues of P^T:", np.round(evals, 4))

idx = np.argmin(np.abs(evals - 1))
vec = np.real(evecs[:, idx])
fixed_row_vector = vec / vec.sum()

print("Stationary (fixed) row vector:", np.round(fixed_row_vector, 4))



## Wrap-up

You just reproduced, in pure Python:
- how a **periodic** chain fails to converge,
- how a **regular** chain always converges to a unique stationary distribution,
- how the shape of a 3-state transition matrix changes a fighter's long-run behavior,
- how to compute a stationary distribution two ways: (1) raising $P$ to a large
  power, and (2) solving for the eigenvector of $P^\top$ with eigenvalue 1.

Next up: `markov_chains_tutorial.md` explains *why* all of this works, and
`markov_chain_new_example.ipynb` applies it to a brand-new (non-ninja) example.
